# Ir Alem 2: Classificação de ECG com Rede Neural

**Projeto CardioIA / Fase 2**

Luiz Felipe Alves Gomes | 2TIAOA 2026/1

## O que esse notebook faz

Treina uma rede neural MLP pra classificar imagens de eletrocardiograma como normal ou anormal. Usa as 233 imagens de ECG coletadas na Fase 1.

In [ ]:
!pip install tensorflow pillow scikit-learn numpy matplotlib -q

## 1. Carregando as imagens

As imagens estao em `data/images/`. Pra rede neural funcionar, cada imagem precisa ter o mesmo tamanho e estar em tons de cinza. Usei 128x128 porque testei com 64x64 primeiro e perdia muito detalhe das ondas do ECG. Depois os pixels viram um vetor de numeros entre 0 e 1.

In [ ]:
import os
import numpy as np
from PIL import Image

IMG_DIR = '../../data/images'
IMG_SIZE = (128, 128)

images = []
labels = []

for filename in sorted(os.listdir(IMG_DIR)):
    filepath = os.path.join(IMG_DIR, filename)

    if filename.startswith('Normal_') or '_normal.' in filename:
        label = 0  # normal
    elif filename.startswith(('Abnormal_', 'MI_', 'History_')) or '_anormal.' in filename:
        label = 1  # anormal
    else:
        continue

    try:
        img = Image.open(filepath).convert('L')  # tons de cinza
        img = img.resize(IMG_SIZE)
        img_array = np.array(img) / 255.0  # normaliza entre 0 e 1
        images.append(img_array.flatten())  # achata pra vetor 1D
        labels.append(label)
    except Exception as e:
        print(f'Erro em {filename}: {e}')

X = np.array(images)
y = np.array(labels)

print(f'Total de imagens: {len(X)}')
print(f'Normal: {sum(y == 0)} | Anormal: {sum(y == 1)}')
print(f'Cada imagem: {IMG_SIZE[0]}x{IMG_SIZE[1]} = {X.shape[1]} pixels')

## 2. Visualizando algumas imagens

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

normais = np.where(y == 0)[0][:5]
for i, idx in enumerate(normais):
    axes[0][i].imshow(X[idx].reshape(IMG_SIZE), cmap='gray')
    axes[0][i].set_title('Normal')
    axes[0][i].axis('off')

anormais = np.where(y == 1)[0][:5]
for i, idx in enumerate(anormais):
    axes[1][i].imshow(X[idx].reshape(IMG_SIZE), cmap='gray')
    axes[1][i].set_title('Anormal')
    axes[1][i].axis('off')

plt.suptitle('ECGs (128x128, tons de cinza)')
plt.tight_layout()
plt.show()

## 3. Treino e teste

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Treino: {len(X_train)} imagens')
print(f'Teste: {len(X_test)} imagens')

## 4. Montando a rede neural

MLP com 3 camadas. BatchNormalization ajuda a estabilizar o treino e Dropout evita que o modelo decore os dados.

- Entrada: 16384 valores (128x128 pixels)
- Camada 1: 128 neuronios + BatchNorm + Dropout 40%
- Camada 2: 64 neuronios + BatchNorm + Dropout 30%
- Saida: 1 neuronio (sigmoid, 0=normal 1=anormal)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Treinando

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=80,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

## 6. Resultado

In [ ]:
from sklearn.metrics import classification_report

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Acuracia no teste: {accuracy:.2%}')
print(f'Loss: {loss:.4f}')

y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int).flatten()

print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Anormal'], zero_division=0))

## 7. Curva de treino

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Treino')
ax1.plot(history.history['val_accuracy'], label='Validação')
ax1.set_title('Acuracia por Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Acuracia')
ax1.legend()

ax2.plot(history.history['loss'], label='Treino')
ax2.plot(history.history['val_loss'], label='Validação')
ax2.set_title('Loss por Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

## 8. O que aprendi

A acuracia ficou baixa comparando com o classificador de texto da Parte 2.

Primeiro testei com imagens 64x64 (4096 pixels por imagem) e o resultado ficou em torno de 42%. Aumentei pra 128x128 (16384 pixels) pra preservar mais detalhe das ondas do ECG, e a acuracia subiu pra ~57%. Melhorou, mas ainda nao e um resultado bom. Mudar a resolucao sozinha nao resolve o problema de fundo.

O dataset mistura imagens de fontes diferentes (JPG do Kaggle + PNG gerados), com estilos visuais bem diferentes. Umas sao fotos de papel, outras sao graficos digitais. Isso confunde o modelo.

O MLP trata cada pixel como independente. Ele nao sabe que o pixel 50 esta do lado do pixel 51. Pra ECG o que importa e o formato das ondas (curvas, picos), nao o valor de cada pixel isolado. Uma CNN entenderia esses padroes espaciais e provavelmente teria resultado bem melhor.

Com 233 imagens o modelo tambem nao tem dados suficientes pra generalizar. Datasets de referencia como o MIT-BIH tem milhares de registros.

O exercicio serviu pra entender o pipeline completo de visao computacional: carregar imagens, pre-processar, montar a rede, treinar e avaliar. E tambem pra ver na pratica que escolher o tipo certo de rede (MLP vs CNN) faz diferenca.